# Trajectory-driven energetic model: derivation and registered results

This notebook derives the energetic quantities reported in the main text and Supplementary Information from publication-safe reconstructed trajectories. It uses no raw audio, pressure, accelerometer, magnetometer, GPS, field-log, or identifying data.

The primary calculation is the trajectory-driven model implemented in `src/dswp/energetics.py`. It is inspired by the hydrodynamic framework of Goldbogen et al. (2019), but it is not a fitted implementation of the analytical prey-interception model. The Aoki et al. (2012) steady-drag calculation is applied to the same observed intervals as additional evidence for the direction of the A/B contrast.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from scipy.stats import ttest_rel

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'pyproject.toml').exists()
)
sys.path.insert(0, str(ROOT / 'src'))

from dswp.aoki import event_aoki_energy_from_trajectories, paired_individual_aoki
from dswp.classification import attach_labels
from dswp.energetics import EnergyParameters, event_energy_from_trajectories, paired_individual_energy
from dswp.figures import plot_individual_energy
from dswp.io import load_buzz_events

EVENTS = ROOT / 'data/dswp/buzz_events.csv'
TRAJECTORY_INDEX = ROOT / 'data/dswp/trajectory_index.csv'

## 1. Publication-safe inputs and fixed parameters

`trajectory_index.csv` points to pseudonymized, local-coordinate `T###.npz` files. Each archive contains one float64 `xyz_m` array. The paths are reconstructed products; they do not contain raw tag channels or absolute locations.

The production model uses one fixed morphology for both tactics. This makes the absolute values model-dependent while preserving a paired, within-whale comparison under identical parameters.

In [ ]:
parameters = EnergyParameters()
parameter_table = pd.Series(asdict(parameters), name='value').to_frame()
trajectory_index = pd.read_csv(TRAJECTORY_INDEX)
release_summary = pd.Series({
    'released_trajectories': len(trajectory_index),
    'individual_tokens': trajectory_index['individual_id'].nunique(),
    'sampling_rates_hz': sorted(trajectory_index['sampling_rate_hz'].unique().tolist()),
})
display(parameter_table)
display(release_summary.to_frame('value'))

## 2. Event interval and energy equations

For each buzz, the first and last acoustic clicks are mapped to the preceding 25-Hz trajectory sample. Speed is the magnitude of successive 3D position differences divided by the sample interval, and tangential acceleration is $a(t)=dV(t)/dt$.

Parasite drag and positive acceleration work are

$$D_{\mathrm{para}}(t)=\gamma_{\mathrm{tail}}\gamma_{\mathrm{depth}}\frac{1}{2}\rho S_{\mathrm{wet}}C_D(t)V(t)^2,$$

$$W_{\mathrm{drag}}=\int D_{\mathrm{para}}(t)V(t)\,dt,$$

$$W_{\mathrm{acc}}=\int\max[0,M(1+k)a(t)V(t)]\,dt.$$

Deceleration is assigned no negative or recovered work. Mechanical translation work is converted to metabolic energy using $\eta_{\mathrm{metab}}\eta_{\mathrm{prop}}$, and rest-of-body metabolism is added over the same first-click--last-click interval.

In [ ]:
events = attach_labels(load_buzz_events(EVENTS), method='gmm')
energy = event_energy_from_trajectories(events, TRAJECTORY_INDEX, parameters=parameters)

efficiency = parameters.eta_metabolic * parameters.eta_propulsive
energy['E_metabolic_drag_J'] = energy['drag_work_J'] / efficiency
energy['E_metabolic_positive_acceleration_J'] = (
    energy['positive_acceleration_work_J'] / efficiency
)
energy.head()

## 3. Component contributions and whale-level aggregation

The paper does not average all buzzes as if they were independent. Type-specific means are first calculated within each individual, pooling repeated deployments, and those paired individual means are then averaged across the 20 whales. The same hierarchy is used for the component table.

In [ ]:
component_columns = [
    'E_metabolic_drag_J',
    'E_metabolic_positive_acceleration_J',
    'E_metabolic_rest_J',
    'E_total_J',
]
component_means_mj = (
    energy.loc[energy['buzz_type'].isin(['A', 'B'])]
    .groupby(['individual_id', 'buzz_type'])[component_columns].mean()
    .groupby('buzz_type').mean()
    .div(1e6)
)
assert np.allclose(component_means_mj.loc['A'], [0.009821, 1.228810, 0.036831, 1.275463], atol=5e-7)
assert np.allclose(component_means_mj.loc['B'], [0.075813, 8.924000, 0.069681, 9.069494], atol=5e-7)
component_means_mj

In [ ]:
paired = paired_individual_energy(energy)
means_mj = paired[['A', 'B']].mean().div(1e6)
paired_test = ttest_rel(paired['A'], paired['B'])
registered_summary = pd.Series({
    'n_paired_individuals': len(paired),
    'mean_A_MJ': means_mj['A'],
    'mean_B_MJ': means_mj['B'],
    'B_to_A_ratio': means_mj['B'] / means_mj['A'],
    'individuals_with_B_above_A': int((paired['B'] > paired['A']).sum()),
    'paired_t_p_value': paired_test.pvalue,
})
assert np.isclose(means_mj['A'], 1.2754630221287353)
assert np.isclose(means_mj['B'], 9.069494397522456)
display(registered_summary.to_frame('value'))
plot_individual_energy(paired)

## 4. Aoki steady-drag comparison

The Aoki formulation is evaluated over exactly the same reconstructed speed samples and event intervals. It is additional evidence in the same direction, not a quantitative calibration of the primary estimates. Different absolute values are expected because it represents active-swimming drag and omits the positive-acceleration and resting terms.

In [ ]:
aoki_events = event_aoki_energy_from_trajectories(events, TRAJECTORY_INDEX)
aoki_paired = paired_individual_aoki(aoki_events, individual_col='individual_id')
aoki_means_mj = aoki_paired[['A', 'B']].mean().div(1e6)
aoki_summary = pd.Series({
    'n_paired_individuals': len(aoki_paired),
    'mean_A_MJ': aoki_means_mj['A'],
    'mean_B_MJ': aoki_means_mj['B'],
    'B_to_A_ratio': aoki_means_mj['B'] / aoki_means_mj['A'],
    'individuals_with_B_above_A': int((aoki_paired['B'] > aoki_paired['A']).sum()),
    'paired_t_p_value': ttest_rel(aoki_paired['A'], aoki_paired['B']).pvalue,
})
assert np.isclose(aoki_means_mj['A'], 0.01307228084262739)
assert np.isclose(aoki_means_mj['B'], 0.11124034997877447)
assert (aoki_paired['B'] > aoki_paired['A']).all()
aoki_summary.to_frame('value')

## Interpretation boundary

The registered primary values are model-estimated metabolic demands over an observable buzz interval. They should be interpreted comparatively. Fixed morphology, differentiation of reconstructed speed, and omission of rotational work, buoyancy, lift, internal fluking work, prey speed, capture success, and prey energy limit the interpretation of absolute values. The Aoki result supports the direction and approximate scale of the tactic contrast under a drag-only formulation; it is not expected to reproduce the primary totals.